In [0]:
%sql
CREATE CATALOG IF NOT EXISTS vinkoscon
MANAGED LOCATION 'abfss://processeddatabricks@strfilese.dfs.core.windows.net/vinkos_managed/';

In [0]:
%sql
-- Forzamos el uso de tu nuevo catálogo
USE CATALOG vinkoscon;

-- Creamos el esquema
CREATE SCHEMA IF NOT EXISTS bronze;

-- Creamos la tabla (CTAS)
CREATE TABLE vinkoscon.bronze.visitas_raw
PARTITIONED BY (create_date)
AS SELECT *  FROM azure_sql_vinkos_catalog.dbo.usuarios_visitas_raw ;



In [0]:
from pyspark.sql.functions import col, regexp_like, lit, trim ,to_timestamp ,to_date, date_format

df = spark.read.table("vinkoscon.bronze.visitas_raw")
emails_valido = df.filter(col("email").rlike(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')).select("*")
sin_duplicados= emails_valido.dropDuplicates(['email'])
 
df_fechalimpia = sin_duplicados.withColumn(
    "fechaPrimeraVisita", 
    to_timestamp("fechaPrimeraVisita", "dd/MM/yyyy HH:mm")
)

# Supongamos que tu columna se llama "fecha_id"
df_fechaformato = df_fechalimpia.withColumn(
    "fechaUltimaVisita", 
    to_date("fechaUltimaVisita", "yyyyMMdd")
)


df_final = df_fechaformato.withColumn("fechaUltimaVisita", date_format("fechaUltimaVisita", "yyyyMMdd"))

display(df_final)

 

In [0]:
%sql
-- Creamos el esquema
CREATE SCHEMA IF NOT EXISTS silver;